In [ ]:
# ============================================================
# CELL 1 - VERIFY LOCAL ENVIRONMENT
# ============================================================

import os
import subprocess
import sys
from pathlib import Path

# Ensure reproducible CuBLAS workspace
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch

project_root = Path.cwd()
if not (project_root / "src" / "dp_forgetbench").exists():
    raise RuntimeError(f"Run this notebook from the repo root, not {project_root}")

print("Project root:", project_root)
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    subprocess.run(["nvidia-smi"])
else:
    print("No CUDA GPU found. Unit tests and tiny smoke tests still run locally.")


In [ ]:
# ============================================================
# CELL 2 - CHECK LOCAL PROJECT FILES
# ============================================================

from pathlib import Path

required = ["src", "tests", "configs", "scripts", "README.md", "requirements.txt"]
for name in required:
    path = Path(name)
    print("OK     " if path.exists() else "MISSING", path)

data_dir = Path("data")
print("\nData directory exists:", data_dir.exists())
if data_dir.exists():
    try:
        print("Data directory entries:", [p.name for p in data_dir.iterdir()])
    except PermissionError as exc:
        print("Data directory exists but is not readable by this process:", exc)


In [ ]:
# ============================================================
# CELL 3 - SET LOCAL IMPORT PATH
# ============================================================

import os
import sys
from pathlib import Path

project_root = Path.cwd()
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ["PYTHONPATH"] = src_path

print("PYTHONPATH:", os.environ["PYTHONPATH"])


In [ ]:
# ============================================================
# CELL 4 - RUN LOCAL TEST SUITE
# ============================================================

import os
from pathlib import Path

# Ensure reproducible CuBLAS workspace and suppress non-deterministic warnings
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Reset TEMP / TMP back to system user temp if previously redirected to local_test_tmp
user_temp = os.environ.get("LOCALAPPDATA", "")
if user_temp and (Path(user_temp) / "Temp").exists():
    os.environ["TEMP"] = str(Path(user_temp) / "Temp")
    os.environ["TMP"] = str(Path(user_temp) / "Temp")

# -p no:cacheprovider avoids Windows permission issues in .pytest_cache.
!python -m pytest -q -p no:cacheprovider


In [ ]:
# ============================================================
# CELL 5 - RUN A LOCAL RESNET-50 FEDERATED TRAINING SMOKE
# ============================================================

import os
import torch

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from dp_forgetbench.data import ClientDataset
from dp_forgetbench.federated import evaluate_loss_accuracy, train_federated

clients = {
    client_id: ClientDataset(
        client_id=client_id,
        x=torch.randn(4, 3, 32, 32),
        y=torch.randint(0, 10, (4,)),
    )
    for client_id in range(3)
}

model, cost = train_federated(
    clients=clients,
    n_features=None,
    federated_config={
        "rounds": 1,
        "local_epochs": 1,
        "local_batch_size": 2,
        "learning_rate": 0.01,
        "client_sample_rate": 1.0,
    },
    privacy_config={"enabled": False},
    seed=1,
    model_config={"name": "groupnorm_resnet50", "num_classes": 10, "base_width": 8},
)

metrics = evaluate_loss_accuracy(model, torch.randn(5, 3, 32, 32), torch.randint(0, 10, (5,)))
print(type(model).__name__)
print(cost.as_dict())
print(metrics)


In [ ]:
# ============================================================
# CELL 6 - LOCAL CIFAR-10 RESNET-50 SMOKE (QUICK CHECK)
# ============================================================

# Runs a quick 2-round smoke test on your local GPU to verify end-to-end training and privacy accounting.
!python scripts/run_multiclass_utility_sweep.py --config configs/multiclass_resnet50_t4_smoke.yaml --output-dir results/local_resnet50_smoke


In [ ]:
# ============================================================
# CELL 7 - LOCAL RESNET-50 DP UTILITY SWEEP (RTX 4060)
# ============================================================

# Runs 40 rounds x 3 epsilon values on your local GPU.
# Expected runtime: ~5-6 min total on RTX 4060 Laptop GPU.
# epsilon=inf  => No DP control  => should reach ~35-45% accuracy  [VALID]
# epsilon=32   => Tight DP       => target ~25-35% accuracy        [WARNING/VALID]
# epsilon=8    => Very tight DP  => may be lower                   [WARNING/INVALID]
!python scripts/run_multiclass_utility_sweep.py --config configs/multiclass_resnet50_local_sweep.yaml


In [ ]:
# ============================================================
# CELL 8 - INSPECT LOCAL RESNET-50 ACCURACY RESULTS
# ============================================================

from pathlib import Path
import pandas as pd

results_path = Path("results/utility_sweep_resnet50/utility_sweep.csv")
if not results_path.exists():
    results_path = Path("results/local_resnet50_smoke/utility_sweep.csv")
if not results_path.exists():
    raise FileNotFoundError("Run cell 6 or cell 7 first; no ResNet-50 results CSV exists yet.")

results = pd.read_csv(results_path)
print("Loaded:", results_path)
cols = [c for c in ["epsilon_target", "epsilon_actual", "seed", "rounds", "test_accuracy", "validity_status"] if c in results.columns]
display(results[cols])
print("Best accuracy:", f"{results['test_accuracy'].max() * 100:.2f}%")
